# UC1 Retrieval + Query Extension Evaluation

This notebook runs the updated retrieval pipeline in `main.py` and evaluates it with the same logic as `2_evaluation_SOTA_NLP_LIR.ipynb` for scientific comparability.

Comparison target:
- Baseline reference (`Bi-Encoder + Cross-Encoder`) from existing UC1 subprocess output
- New query extension (`Bi-Encoder + Cross-Encoder`) from updated pipeline

In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd

project_root = Path.cwd()
while project_root.name != "Legal-Query-Synthesis-from-Business-Processes-for-Agentic-Retrieval" and project_root.parent != project_root:
    project_root = project_root.parent

os.chdir(project_root)
print(f"Working directory: {Path.cwd()}")

In [ ]:
from main import retrieval_pipeline

# Runs UC1 medium/subprocess retrieval with baseline + query extension
# Output: output_ranking.xlsx and output_with_agents.csv in project root
retrieval_pipeline(provider="gemini")

In [ ]:
def clean_text(text):
    cleaned_text = str(text).replace("or\n\n\n", " ")
    cleaned_text = cleaned_text.replace("or\n\n", " ")
    cleaned_text = cleaned_text.replace("and\n\n\n", " ")
    cleaned_text = cleaned_text.replace("and\n\n", " ")
    cleaned_text = cleaned_text.replace("\n\n\n", " ")
    cleaned_text = cleaned_text.replace("\n\n", " ")
    cleaned_text = cleaned_text.replace("\n \n", " ")
    cleaned_text = cleaned_text.replace("\n", " ")
    return cleaned_text


def evaluate_with_original_logic(df_gs, df_alg):
    df_gs = df_gs.copy()
    df_alg = df_alg.copy()

    df_gs["query_cleaned"] = df_gs.apply(lambda row: clean_text(row["query"]), axis=1)
    df_gs["rel_text_cleaned"] = df_gs.apply(lambda row: clean_text(row["rel_text"]), axis=1)
    df_gs = df_gs.drop(["query", "rel_text"], axis=1)
    df_gs = df_gs.rename(columns={"query_cleaned": "query", "rel_text_cleaned": "rel_text"})

    df_alg["query"] = df_alg["query"].apply(clean_text)
    df_alg["rel_text"] = df_alg["rel_text"].apply(clean_text)
    df_alg["rank"] = df_alg.groupby("query")["score"].rank(ascending=False)

    df_gs_enhanced = pd.merge(
        df_gs,
        df_alg,
        how="left",
        left_on=["query", "rel_text"],
        right_on=["query", "rel_text"],
    )

    df_gs_enhanced["AP"] = 1 / df_gs_enhanced["rank"]
    df_gs_enhanced = df_gs_enhanced.fillna(0)

    map_value = df_gs_enhanced["AP"].mean()
    tp_per_query = df_gs_enhanced.groupby("query")["rank"].apply(lambda x: (x > 0).sum()).reset_index(name="count")
    fn_per_query = df_gs_enhanced.groupby("query")["rank"].apply(lambda x: (x == 0).sum()).reset_index(name="count")

    return {
        "map": map_value,
        "avg_tp": tp_per_query["count"].mean(),
        "avg_fn": fn_per_query["count"].mean(),
        "details": df_gs_enhanced,
    }

In [ ]:
gs_path = project_root / "regulatory_relevance4process-D73C/SOTA_NLP_LIR/output_ranking_input_eval/uc1/gold_standard/gs_uc1_subprocess_level.xlsx"
baseline_reference_path = project_root / "regulatory_relevance4process-D73C/SOTA_NLP_LIR/output_ranking_input_eval/uc1/algo_output/uc1_subprocess_level_algo_output_bi_ce.xlsx"
new_output_path = project_root / "output_ranking.xlsx"

# Reference baseline (original SOTA notebook output)
df_gs = pd.read_excel(gs_path)
df_baseline_reference = pd.read_excel(baseline_reference_path)

# New query extension output from updated pipeline
df_query_extension = pd.read_excel(new_output_path, sheet_name="Bi_CE_query_extension")

reference_eval = evaluate_with_original_logic(df_gs, df_baseline_reference)
query_extension_eval = evaluate_with_original_logic(df_gs, df_query_extension)

comparison = pd.DataFrame(
    [
        {
            "run": "baseline_reference_bi_ce",
            "MAP": reference_eval["map"],
            "avg_true_positives": reference_eval["avg_tp"],
            "avg_false_negatives": reference_eval["avg_fn"],
        },
        {
            "run": "query_extension_bi_ce",
            "MAP": query_extension_eval["map"],
            "avg_true_positives": query_extension_eval["avg_tp"],
            "avg_false_negatives": query_extension_eval["avg_fn"],
        },
    ]
)
comparison

In [ ]:
results_path = project_root / "regulatory_relevance4process-D73C/SOTA_NLP_LIR/output_ranking_input_eval/uc1/results_query_extension_subprocess_uc1_bi_ce.xlsx"
query_extension_eval["details"].to_excel(results_path, index=False)
comparison